# API Advanced using the tilburg.ai API platform

In this tutorial, we’ll gradually explore how to use the **tilburg.ai API** for marketing analytics in Python.
Each section builds on the previous one:

1. Getting started – setting up and sending your first message  
2. Basic uses of model responses  
3. Embeddings and semantic similarity

Every section ends with a short **exercise** so that you can experiment with
your own prompts and parameters.

# 1. Getting Started

Let’s begin by setting up access to the API and making our very first request.
We will use small campaign and customer examples so that each API concept connects to a marketing-analytics task.

## 1.1 Generate your API key 

Log in to [tilburg.ai api-platform](https://api-portal.tilburg.ai/) with your Tilburg University account. Click on **+ Create API key** and give the key a descriptive name, such as "ODCM Tutorial".

Now you will be shown an API key: "sk-......", copy this key. The API key is your personal password for talking to different AI servers. To keep it safe, we store it in a hidden file called `.env` in the same folder as this notebook.

Example of what that file should contain:

```
TILBURG_AI_API_KEY=sk-XXXXXXXXXXXXXXXXXXXX
```

The code below loads the key from `.env` and prints only the beginning.

In [ ]:
from dotenv import find_dotenv, load_dotenv
import os

found = find_dotenv(usecwd=True)
load_dotenv(found, override=True)
api_key = os.getenv("TILBURG_AI_API_KEY")
if not api_key:
    raise RuntimeError("TILBURG_AI_API_KEY was not found. Check your .env file.")

print("Found .env at:", found)
print("API key loaded successfully.")

### 🧩 Exercise 1 — Checking your setup

- Verify that the path printed above actually points to your own project folder.  
- If it shows a different directory (e.g. your user home), move your `.env` file here.  

### Suggested solution

The path should point to the folder containing your notebook and `.env` file. The final line should confirm that the key was loaded successfully. Never print the key itself.

## 1.2 Your first model response using the Python SDK

With the key loaded, we should choose a model and start saying "Hello" to it. To do this we will use Python SDK, setup documentation can be found at [tilburg.ai api-platform getting started](https://api-portal.tilburg.ai/getting-started/python-sdk).

First, install the OpenAI Python package and `python-dotenv` in your terminal.

```
pip install openai==2.50.0 python-dotenv
```

In [ ]:
%pip install openai==2.50.0 python-dotenv

In [ ]:
from openai import OpenAI
from dotenv import find_dotenv, load_dotenv
import os

# Setup endpoint and gather API key 
API_ENDPOINT = "https://api.tilburg.ai/v1"
found = find_dotenv(usecwd=True)
load_dotenv(found, override=True)
API_KEY = os.getenv("TILBURG_AI_API_KEY")

client = OpenAI(
    api_key=API_KEY,
    base_url=API_ENDPOINT,
)

model = "gpt-5.4-nano"

response = client.responses.create(
    model=model,
    instructions="Keep the answer short and do not use styling",
    input="Which customer segment is most likely to respond to a discount campaign, and why?",
    temperature=2
)

print(response.output_text)


### 🧩 Exercise 2 — Your first modification
- Run the starter code to check if it works
- Rerun the starter code. Is the output the same as before? Why is it, or is it not?
- Change the capital "Which" to lowercase and rerun. Is the output the same as before? Why is it, or is it not?
- Replace the prompt with your own marketing question (for example *“Which customers should receive a retention offer?”*).  
- Change the `temperature` value (try `0`, `1`, and `1.5`) and observe how the creativity of the response changes.  
- Reflect: what does “temperature” mean for text generation?

### Suggested solution

A suitable marketing prompt could be `Which customer segment is most likely to respond to a loyalty discount? Explain your reasoning briefly.` With a lower temperature, the answer is generally more consistent; with a higher temperature, wording and recommendations may vary more. Temperature does not make the analysis causal or guarantee a better answer.

## 1.3 Using reasoning for multi-step tasks

Some models can spend additional computation on tasks that involve multiple steps, such as calculations, comparisons, or planning. We control this with the `reasoning` parameter. Higher reasoning effort may improve difficult answers, but it can also increase response time and cost.

The model should provide a concise explanation or summary of the result—not its private chain-of-thought.

In [ ]:
reasoning_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions=(
        "Solve the problem carefully. Give the final answer and a short explanation; "
        "do not reveal hidden chain-of-thought."
    ),
    input=(
        "A retailer runs Search, Social, and Email campaigns. Allocate each "
        "customer's revenue using last-touch attribution, calculate ROAS and "
        "conversion rate for each campaign, exclude campaigns with fewer than 50 "
        "touchpoints, and recommend where to increase budget. Search has spend "
        "1200, 80 touchpoints, 24 conversions, and 3600 revenue. Social has "
        "spend 900, 140 touchpoints, 21 conversions, and 3150 revenue. Email "
        "has spend 300, 60 touchpoints, 18 conversions, and 2700 revenue. "
        "State one limitation: these figures do not establish causal attribution."
    ),
    reasoning={"effort": "medium"}
)

print(reasoning_response.output_text)

### 🧩 Exercise 3 — Comparing reasoning effort on a marketing problem

Use the code below to compare different reasoning-effort settings.

- Does low effort omit any requested metric, filtering rule, or limitation?
- Does higher effort improve the completeness of the explanation?
- Does it increase response time?
- Is the additional effort justified for this marketing-analysis task?

If the model available to you does not support the `reasoning` parameter, choose a reasoning-capable model listed on the tilburg.ai API platform.

### Suggested solution

The calculations are ROAS = revenue / spend and conversion rate = conversions / touchpoints. Search has ROAS 3.0 and a 30% conversion rate; Social has ROAS 3.5 and a 15% conversion rate; Email has ROAS 9.0 and a 30% conversion rate. All campaigns pass the 50-touchpoint filter, so Email has the highest observed ROAS. A careful recommendation should note that last-touch attribution and these observational figures do not establish causal effects. Low effort may omit one of these requested steps; higher effort may be more complete, but always check the numbers yourself.

In [ ]:
problem = (
    "A retailer runs Search, Social, and Email campaigns. Allocate each "
    "customer's revenue using last-touch attribution, calculate ROAS and "
    "conversion rate for each campaign, exclude campaigns with fewer than 50 "
    "touchpoints, and recommend where to increase budget. Search has spend "
    "1200, 80 touchpoints, 24 conversions, and 3600 revenue. Social has "
    "spend 900, 140 touchpoints, 21 conversions, and 3150 revenue. Email "
    "has spend 300, 60 touchpoints, 18 conversions, and 2700 revenue. "
    "State one limitation: these figures do not establish causal attribution."
)

for effort in ["low", "medium", "high"]:
    response = client.responses.create(
        model="gpt-5.4-nano",
        instructions=(
            "Solve the problem carefully. Provide the final answer and a brief "
            "summary of the main calculation, without providing hidden chain-of-thought."
        ),
        input=problem,
        reasoning={"effort": effort}
    )

    print(f"\nReasoning effort: {effort}")
    print(response.output_text)

# 2. Basic Uses of Model Responses

Once you can send a single message, the next step is learning how to
*structure a conversation* or repeat a task for many inputs.
We’ll start with a small dataset of customer feedback about marketing campaigns.

In [ ]:
reviews = [
    {"review": "The spring-sale ad made the product feel affordable and relevant."},
    {"review": "The retargeting emails arrived too often and annoyed me."},
    {"review": "The social-media video clearly explained the product's benefits."},
    {"review": "The discount looked attractive, but the landing page was confusing."},
    {"review": "The campaign felt authentic and made me consider purchasing."}
]

for i, item in enumerate(reviews, 1):
    print(f"{i}. {item['review']}")

- Each element here is a small dictionary (key → value pair).  
- This mirrors the structure of JSON data returned by APIs — which is why
we practice reading and writing lists of dictionaries instead of spreadsheets.

## 2.1 Asking the model to analyse many texts

We’ll now send each piece of campaign feedback to the API and ask for a quick sentiment score
between 1 (very negative) and 5 (very positive).

In [ ]:
import time

for i, item in enumerate(reviews, 1):
    response = client.responses.create(
        model="gpt-5.4-nano",
        instructions="Rate the emotional tone of this marketing-campaign feedback from 1 (very negative) to 5 (very positive). Return only the number.",
        input=f"Review: {item['review']}"
    )
    
    item["score"] = response.output_text.strip()
    print(f"{i}. {item['score']}")
    time.sleep(0.5)


### 🧩 Exercise 4 — Experimenting with prompts

- Change the instruction so that the model returns both a **score and one-sentence explanation**.  
- What happens if you set `temperature=1`?  
- Try writing a version that asks the model to output `"positive"`, `"neutral"`, or `"negative"` instead of numbers.

### Suggested solution

Use an instruction such as `Return a score from 1 to 5 followed by one sentence explaining the likely effect on purchase intent.` For a three-class version, use `Return exactly one label: positive, neutral, or negative.` Exact scores can vary between runs, so focus on whether the output follows the requested format and reflects the campaign feedback.

## 2.2 Does the model remember?

Each API call is independent — the model does *not* remember previous messages unless you send them again.
Let’s see that in action.

In [ ]:
r1 = client.responses.create(
    model="gpt-5.4-nano",
    input="The customer is interested in the spring-sale campaign.",
    temperature=0
)
print("First answer:", r1.output_text, "\n")

r2 = client.responses.create(
    model="gpt-5.4-nano",
    input="Which campaign is the customer interested in?",
    temperature=0
)
print("Second answer:", r2.output_text)

### 🧩 Exercise 5 — Building a short conversation

- Modify the code so that the **second request includes both inputs** in its `input` list.  
- Observe how the answer changes when the model has access to the conversation history.

### Suggested solution

The second response should identify the spring-sale campaign when both inputs are supplied. The first pair of requests should not reliably carry information from the first request into the second one.

In [ ]:
r2 = client.responses.create(
    model="gpt-5.4-nano",
    input=[
        {"role": "user", "content": "The customer is interested in the spring-sale campaign."},
        {"role": "user", "content": "Which campaign is the customer interested in?"}
    ],
    temperature=0
)
print("answer:", r2.output_text)

# 3. Embeddings — Understanding Similarity

While model responses produce *content*, embeddings represent *meaning* as numbers.
They allow us to measure how similar two pieces of text are, even if they use different words.
This section shows two simple applications.

In [ ]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text,
    )
    embedding = response.data[0].embedding

    return embedding

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    mag = (sum(x*x for x in a)**0.5) * (sum(y*y for y in b)**0.5)
    return dot / mag

## 3.1 Application 1 – Semantic Search

Suppose we have a few marketing messages and a search query from a customer.
We’ll embed both and sort the messages by semantic similarity.

In [ ]:
products = [
    {"desc": "Save 20% on sustainable running shoes this weekend"},
    {"desc": "Free delivery on your next home-fitness equipment order"},
    {"desc": "New summer collection: lightweight jackets for commuters"}
]

for p in products:
    p["embedding"] = get_embedding(p["desc"])

query = "discount for environmentally conscious runners"
query_emb = get_embedding(query)

for p in products:
    p["similarity"] = cosine(p["embedding"], query_emb)

products.sort(key=lambda x: x["similarity"], reverse=True)
for p in products:
    print(p["desc"], "→", round(p["similarity"], 3))

### 🧩 Exercise 6 — New products

- Add one more description to the `products` list.  
- Try a new `query` such as *“free shipping for home workouts”* or *“clothing for cycling to work”*.  
- Which description is most similar?  
- Reflect on how embeddings capture *meaning*, not exact wording.

### Suggested solution

Add a marketing message as another dictionary, for example `{'desc': 'Limited-time offer for trail runners who prefer recycled materials'}`. For the supplied query, the sustainable running-shoes message should usually rank highly, but similarity scores and rankings are model-dependent.

## 3.2 Application 2 – Automatic Coding of Survey Responses

Instead of hand-coding open answers, we can embed marketing-survey responses and category descriptions and match them by similarity.

In [ ]:
responses = [
    {"text": "The campaign made it easy to discover products I needed."},
    {"text": "The offer was too expensive for the value provided."},
    {"text": "The brand's influencer content felt authentic and trustworthy."}
]

themes = [
    {"theme": "Discovery", "desc": "How easily marketing helps customers discover relevant products."},
    {"theme": "Price", "desc": "Affordability, discounts, and value for money."},
    {"theme": "Authenticity", "desc": "Whether brand communication feels genuine and trustworthy."}
]

for r in responses:
    r["embedding"] = get_embedding(r["text"])
for t in themes:
    t["embedding"] = get_embedding(t["desc"])

for r in responses:
    best = max(themes, key=lambda t: cosine(r["embedding"], t["embedding"]))
    r["best_theme"] = best["theme"]
    print(r["text"], "→", r["best_theme"])

### 🧩 Exercise 7 — Expanding categories

- Add a new theme such as `"Speed"` or `"Design"`.  
- Try new responses in different wording.  
- Observe whether the model still assigns them to the correct theme.
- Remember that similarity matching is a heuristic, not a guarantee that the assigned theme is correct.

### Suggested solution

Add a theme with a marketing definition, such as `{'theme': 'Creative', 'desc': 'Opinions about the visual style or originality of an advertisement.'}`. Check both the assigned label and whether the category definitions overlap; revise the definitions if the classifications are ambiguous.

# 4. Wrapping Up

In this tutorial we:

- Set up API access with tilburg.ai
- Sent and modified chat prompts  
- Used loops to analyse multiple texts  
- Created embeddings to measure meaning

You now know how to combine these building blocks for your own projects:
from chat assistants to recommender systems and data analysis pipelines.

## Privacy, cost, and limitations

- API requests may incur costs. Check the current model pricing and monitor your usage.
- Do not send confidential, personal, or otherwise sensitive research data unless this is permitted by your institution and the relevant data-management agreements.
- Model outputs can be incorrect or inconsistent. Treat them as suggestions and validate results, especially when coding research data.
- Model behaviour and availability can change over time, so record the model name, prompts, and other settings when you need reproducible results.